Notebook to practice to construct a nice TPM data processing pipeline.

In [1]:
import pandas as pd
import numpy as np
import os
import glob
import sys

In [2]:
fcnts_path = "C:/Users/eddyk/OneDrive/Documents/vanopijnen_lab/pipeline_practice/Fcnts"

# Store folder name
files = os.listdir(fcnts_path)

# Filter out the summary files and keep only NDC comparisons
files = [csv for csv in files if ".summary" not in csv and "NDC0hr" in csv]

# Attach the folder to each of the files
files = ["".join([fcnts_path, "/" , csv]) for csv in files]

# Load each file as a dataframe
dfs = [pd.read_table(csv, sep = "\t", header = 0, skiprows = 1) for csv in files]

# Rename the columns of each pandas dataframe




In [21]:
practice_df = dfs[0]
practice_df.head(n = 5)



# Subset to the columns that either start with a "/" or are named "Genedid"





# Remember, we only need to get the NDC0hr columns from the 1st thing??
# Lowkey just process everything the same, then remove duplicate columns at the end to get rid of all the NDC?





# Set Gene names to the row names
practice_df = practice_df.set_index("Geneid")

# Select length column and those that correspond to the Fcnts values
practice_df = practice_df.loc[:, [col for col in list(practice_df.columns) if col == "Length" or col.startswith("/")]]

# Convert every entry of the dataframe to an integer to allow for integer manipulation
practice_df = practice_df.apply(lambda column: [int(entry) for entry in column])

# Divide Length by 1000 to get kb
practice_df["Length"] = practice_df["Length"].apply(lambda column: column / 1000)

# Select only columns with Fcnts
fcnts_cols = [col for col in list(practice_df.columns) if col != "Length"]

# Fcnts / gene length = (Counts per kb)
practice_df[fcnts_cols] = practice_df[fcnts_cols].apply(lambda column: column / practice_df["Length"])

# (Counts per kb) * 10^6 / (Total counts/kb) = TPM
practice_df[fcnts_cols] = practice_df[fcnts_cols].apply(lambda column: column * 10**6 / sum(column))

# Remove length column
practice_df.drop(columns = "Length", inplace = True)

# Rename the columns to remove extra crap

# can use df.rename(lambda x: ) to rename the samples?

# Obtain index of the last instance of / in the name of the thing
# Remove ".bam"
def sample_name_strip(name):
    """
    
    """
    # Find index of the last / 
    samplename_start_idx = name.strip().rfind("/") + 1
    
    # Subset 
    new_name = name[samplename_start_idx:]

    # Find index of . (.bam is at end of sample name) and remove filetag
    filetag_start_idx = new_name.rfind(".")
    new_name = new_name[:filetag_start_idx]
    
    # Remove "T4-wt"
    new_name = new_name.replace("T4-wt", "")

    return new_name

# Apply stripper to each column names
practice_df = practice_df.rename(columns = lambda column: sample_name_strip(column))

In [22]:

practice_df

,12CEF12CIP1hr-a,12CEF12CIP1hr-b,12CEF12CIP1hr-c,NDC0hr-a,NDC0hr-b,NDC0hr-c
Geneid,,,,,,
SP_0001,114.547926,109.150760,113.862974,102.614861,108.551217,95.498504
SP_0002,73.295354,102.588865,84.181428,83.732058,105.322135,93.891570
SP_0003,50.592072,88.178438,72.986338,54.760090,46.243152,42.709114
SP_0004,161.505461,149.196178,190.529657,170.615323,182.004097,179.954974
SP_0005,34.001150,39.322556,37.728815,26.057362,35.553683,30.138456
...,...,...,...,...,...,...
SP_2236,24.846994,18.283227,19.416237,23.136758,23.966926,31.771686
SP_2237,7.690736,0.000000,14.423491,1.932435,0.000000,6.492411
SP_2238,24.225819,41.930684,33.444469,14.203398,28.786362,21.303224
